# M14 lab: did it work? Regression and difference-in-differences on simulated panels

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M14 · about 60 minutes, plus 90 minutes of practice at the end.

**Every dataset in this notebook is simulated.** The homes, the tariff, the regions and the policy are made up by a seeded generator, with a true effect set in advance. That is the point: real studies never see the truth, so here you can score each method against it.

You will rebuild the lesson's two panels in NumPy and check them against the page with `assert` cells, then go further than a browser can:

1. The generator and the tariff panel: 400 homes, 24 months, a time-of-use (TOU) tariff from month 13 for homes that opt in.
2. Four estimates against the truth: a naive gap, a regression with controls, before–after, difference-in-differences (DiD).
3. The two-way fixed-effects regression, and standard errors clustered by home.
4. An event study, and four threats: selection on trends, anticipation, spillover, confounding.
5. 200 redrawn panels: how often does each interval hold the truth?
6. The regions panel: a policy, solar investment, a boom and projects that move across a border.

**You need** Python as in lesson B2 (NumPy arrays, a `for` loop) and statistics as in lesson K5 (means, standard errors). Least squares is explained where it is first used, in section 2; lesson M10 goes further with it.

**How to run it.** Put this notebook and `m14-did-it-work.json` (*Lesson settings*) in one folder. In Google Colab: *File → Upload notebook*, then upload the settings file with the folder icon on the left, then *Runtime → Run all*. Locally: Python 3.9 or later with `numpy`; `matplotlib` is optional.

**Tested.** On 6 November 2026 with Python 3.9 and NumPy 2.0, the whole notebook ran in under 5 seconds on a 2025 laptop (Apple M5); a free Colab session may take several times longer. On a Mac, NumPy 2.0 can print harmless "divide by zero encountered in matmul" warnings from Apple's maths library; the asserts still check every number.

In [ ]:
import json
from pathlib import Path

import numpy as np

assert Path("m14-did-it-work.json").exists(), "m14-did-it-work.json not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D = json.load(open("m14-did-it-work.json"))
REF = D["reference"]
T, R = D["tou"], D["regions"]
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib not installed: the charts are skipped; every number is still printed.")

def close(a, b, rel=1e-8):
    # the page's reference block keeps 10 significant digits
    return abs(a - b) <= rel * max(1.0, abs(b))
print(D["about"][:160], "...")

## 1. The generator and the tariff panel

The page draws with **mulberry32**, a small 32-bit generator (lesson M7 ports it too), and makes an approximately normal draw as the sum of 12 uniform draws minus 6 (the Irwin–Hall distribution: mean 0, standard deviation 1). Only $+ - \times \div$ are used, so Python reproduces the browser's panel to the last digit. For your own work, use NumPy's `default_rng`, as section 5 does.

For each home, in this order: air conditioning (a draw below 0.6), size in thousand square feet (1 to 3), engagement $e$ (normal, **never observed by an analyst**), selection noise $z$ (normal); then one noise draw per month.

- **Opt in** when $e - 0.6\,(s - 2) + 0.6\,z > 1$: engaged owners and smaller homes sign up more.
- **True effect** $\tau_i = \min(0, -(0.20 + 0.08\,e))$ kW from month 13, in homes that opted in.
- **Evening peak** $y_{it} = 2.5 + 0.6\,ac + 0.35\,(s - 2) - 0.25\,e + \text{season} + ac \cdot \text{AC season} \cdot \text{cooling} + 0.3\,\varepsilon + \tau_i D_{it}$, in kW; the second summer is cooler (cooling 0.75). The 2.5 keeps every home-month above zero: a home's evening peak cannot be negative (the cell below checks it).

In [ ]:
MASK = 0xFFFFFFFF

def imul(a, b):
    # JavaScript's Math.imul: the low 32 bits of a * b, read as an unsigned number here
    return (a * b) & MASK

def mulberry32(seed):
    a = seed & MASK
    def draw():
        nonlocal a
        a = (a + 0x6D2B79F5) & MASK
        t = imul(a ^ (a >> 15), a | 1)
        t ^= (t + imul(t ^ (t >> 7), t | 61)) & MASK
        return ((t ^ (t >> 14)) & MASK) / 4294967296
    return draw

def gauss(r):
    s = 0.0
    for _ in range(12):
        s += r()
    return s - 6

def tou_panel(T, threat=None):
    # the page's touPanel, line for line: returns per-home arrays and the outcome matrix y[home, month]
    r, X, t0 = mulberry32(T["seed"]), T["threats"], T["start"]
    ac, size, eng, on, tau, Y = [], [], [], [], [], []
    for i in range(T["n"]):
        a_c = 1 if r() < T["acShare"] else 0
        s = T["size"][0] + (T["size"][1] - T["size"][0]) * r()
        e = gauss(r)
        z = gauss(r)
        o = 1 if e + T["select"]["size"] * (s - 2) + T["select"]["noise"] * z > T["select"]["cut"] else 0
        ta = min(0, -(T["effect"]["base"] + T["effect"]["engaged"] * e))
        lev = T["level"]["base"] + T["level"]["ac"] * a_c + T["level"]["size"] * (s - 2) + T["level"]["engaged"] * e
        y = []
        for t in range(T["months"]):
            m, yr = t % 12, (0 if t < 12 else 1)
            v = lev + T["season"][m] + a_c * T["acSeason"][m] * T["cooling"][yr] + T["noise"] * gauss(r)
            if o and t >= t0: v += ta
            if threat == "trend" and o: v += X["trend"] * (t - (t0 - 1))
            if threat == "anticipate" and o and t0 - X["anticipateMonths"] <= t < t0: v += X["anticipateShare"] * ta
            if threat == "spill" and not o and t >= t0: v += X["spill"]
            if threat == "bundle" and o and t >= t0 and e > X["bundleCut"]: v += X["bundle"]
            y.append(v)
        ac.append(a_c); size.append(s); eng.append(e); on.append(o); tau.append(ta); Y.append(y)
    return dict(ac=np.array(ac), size=np.array(size), e=np.array(eng), on=np.array(on, dtype=bool), tau=np.array(tau), y=np.array(Y))

P = tou_panel(T)
y, on = P["y"], P["on"]
print(f"{on.sum()} of {len(on)} homes opted in; panel shape {y.shape} (homes, months)")
assert on.sum() == REF["tou"]["nOn"] and (~on).sum() == REF["tou"]["nOff"]
assert close(y[0, 0], REF["tou"]["firstY"]) and close(y[-1, 23], REF["tou"]["lastY"]), "the generator differs from the page's"
assert (y > 0).all(), "a home's evening peak cannot be negative"
print(f"evening peaks from {y.min():.2f} to {y.max():.2f} kW (median {np.median(y):.2f})")
ATT, ATE = P["tau"][on].mean(), P["tau"].mean()
assert close(ATT, REF["tou"]["att"]) and close(ATE, REF["tou"]["ate"])
print(f"truth on those who opted in (ATT) {ATT:.3f} kW; if every home were on the tariff (ATE) {ATE:.3f} kW")

## 2. Four estimates against the truth

- **Naive:** TOU homes against flat homes, months 13–24.
- **Regression with controls:** each home's after-period mean on a 0/1 tariff flag (a *dummy*) plus home size (and air conditioning), the *controls*.
- **Before–after:** TOU homes after against before.
- **DiD:** the change in TOU homes minus the change in flat homes, with the standard error of a difference of two means of home-level changes.

**Least squares in brief.** A regression writes each home's after-period peak as $b_0 + b_1\,\text{TOU}_i + b_2\,\text{size}_i$ and picks the $b$'s that make the sum of squared misses, $\sum_i (y_i - b_0 - b_1 \text{TOU}_i - b_2 \text{size}_i)^2$, as small as possible. In NumPy, `np.linalg.lstsq(X, y)` does it: each row of `X` is one home (a 1 for $b_0$, its 0/1 flag, its size) and it returns the $b$'s. $b_1$, the flag's coefficient, is the fitted gap between TOU and flat homes at equal size. With the flag alone it is exactly the naive gap: the cell checks this.

In [ ]:
t0 = T["start"]
pre, post = y[:, :t0].mean(axis=1), y[:, t0:].mean(axis=1)
chg = post - pre

def diff_means(a, b):
    est = a.mean() - b.mean()
    se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    return est, se

naive = diff_means(post[on], post[~on])
ba = (chg[on].mean(), np.sqrt(chg[on].var(ddof=1) / on.sum()))
did = diff_means(chg[on], chg[~on])

def ols(X, yv):
    return np.linalg.lstsq(X, yv, rcond=None)[0]

ones = np.ones(len(on))
assert abs(ols(np.column_stack([ones, on]), post)[1] - naive[0]) < 1e-9   # the flag alone: the naive gap
reg_size = ols(np.column_stack([ones, on, P["size"]]), post)[1]
reg_both = ols(np.column_stack([ones, on, P["size"], P["ac"]]), post)[1]

for name, (est, se) in [("naive", naive), ("before-after", ba), ("DiD", did)]:
    lo, hi = est - 1.96 * se, est + 1.96 * se
    print(f"{name:13s} {est:+.3f} kW  (about 95%: {lo:+.3f} to {hi:+.3f}; holds the truth: {lo <= ATT <= hi})")
print(f"regression with size {reg_size:+.3f} kW; with size and air conditioning {reg_both:+.3f} kW; truth {ATT:+.3f}")

assert close(naive[0], REF["tou"]["naive"][0]) and close(naive[1], REF["tou"]["naive"][1])
assert close(ba[0], REF["tou"]["beforeAfter"][0]) and close(did[0], REF["tou"]["did"][0]) and close(did[1], REF["tou"]["did"][1])
assert close(reg_size, REF["tou"]["regSize"]) and close(reg_both, REF["tou"]["regBoth"])

## 3. The two-way fixed-effects regression, and clustered standard errors

The regression form of DiD gives each home its own level $\alpha_i$ and each month its own level $\lambda_t$:

$$y_{it} = \alpha_i + \lambda_t + \tau D_{it} + \varepsilon_{it}.$$

With one start date and every home seen every month, $\hat\tau$ equals the DiD of means. A home's 24 months are not 24 independent draws, so the standard error must treat the **home** as the unit: the cluster-robust (sandwich) estimator below. Compare it with the page's two-sample formula, and with the naive formula that treats every home-month as independent.

In [ ]:
n, m = y.shape
yv = y.ravel()                                   # row k = home k // m, month k % m
home, month = np.repeat(np.arange(n), m), np.tile(np.arange(m), n)
Dv = (on[home] & (month >= t0)).astype(float)
X = np.zeros((n * m, n + m))                     # home dummies, then month dummies (month 1 dropped)
X[np.arange(n * m), home] = 1
X[np.arange(n * m), n + month] = 1
X = np.column_stack([Dv, X[:, :n], X[:, n + 1:]])
b, *_ = np.linalg.lstsq(X, yv, rcond=None)
tau_twfe = b[0]
print(f"two-way fixed effects: {tau_twfe:+.6f} kW;  DiD of means: {did[0]:+.6f} kW")
assert abs(tau_twfe - did[0]) < 1e-9

# standard errors by the within transformation (Frisch-Waugh-Lovell): remove each home's and each month's mean from y and D;
# the residuals and tau are those of the full regression
def three_se(y, on, t0):
    # tau and three standard errors for a balanced panel y[home, month] with one start month t0
    n, m = y.shape
    Dm = (on[:, None] & (np.arange(m)[None, :] >= t0)).astype(float)
    dd = (Dm - Dm.mean(1, keepdims=True) - Dm.mean(0, keepdims=True) + Dm.mean()).ravel()
    yy = (y - y.mean(1, keepdims=True) - y.mean(0, keepdims=True) + y.mean()).ravel()
    tau = dd @ yy / (dd @ dd)
    u = yy - tau * dd
    hid = np.repeat(np.arange(n), m)
    chg = y[:, t0:].mean(1) - y[:, :t0].mean(1)
    return {"tau": tau,
            "two-sample": diff_means(chg[on], chg[~on])[1],                                       # the page's formula
            "clustered by home": np.sqrt(np.sum(np.bincount(hid, weights=dd * u) ** 2) * n / (n - 1)) / (dd @ dd),
            "rows as independent": np.sqrt(u @ u / (n * m - n - m) / (dd @ dd))}                 # every home-month alone

S3 = three_se(y, on, t0)
assert abs(S3["tau"] - tau_twfe) < 1e-9 and abs(S3["two-sample"] - did[1]) < 1e-12
se_cluster, se_naive = S3["clustered by home"], S3["rows as independent"]
print(f"standard error: page's two-sample {did[1]:.4f}, clustered by home {se_cluster:.4f}, rows as independent {se_naive:.4f}")

The clustered and the page's two-sample standard errors agree closely. The row-by-row formula assumes every home-month is independent; in this simulation each month's noise is drawn afresh, so it happens to err on the wide side here. Real homes have habits that persist from month to month, and then the row-by-row formula understates the uncertainty (practice task 3). **Cluster by the unit that was assigned: here, the home.**

## 4. An event study, and the four threats

For each month, the DiD of that month against month 12: that month's TOU–flat gap minus the month-12 gap, the **change in the gap**. Zero means the gap stayed the same. Before the start the estimates should sit near zero; similar earlier trends **support** the comparison, they never prove the after-period counterfactual.

The cell computes it month by month (as the page does), prints every month's estimate and interval, then fits the **event-study regression**, $y_{it} = \alpha_i + \lambda_t + \sum_{k \ne -1} \beta_k\,\mathbf{1}[\text{TOU}_i,\ t - 13 = k] + \varepsilon_{it}$, and checks that with one start date and a balanced panel its $\beta_k$ equal the month-by-month numbers.

In [ ]:
def event_study(y, on, ref):
    out = []
    for t in range(y.shape[1]):
        if t == ref:
            out.append((0.0, 0.0)); continue
        out.append(diff_means(y[on, t] - y[on, ref], y[~on, t] - y[~on, ref]))
    return np.array(out)

ES = event_study(y, on, t0 - 1)
assert np.allclose(ES[:, 0], REF["tou"]["eventStudy"], rtol=1e-8, atol=1e-12)
print("month   k   change in the gap, kW   about-95% interval")
for t in range(m):
    est, se = ES[t]
    band = "reference" if t == t0 - 1 else f"{est - 1.96 * se:+.3f} to {est + 1.96 * se:+.3f}"
    print(f"{t + 1:5d} {t - t0:+4d}   {est:+.3f}                  {band}")
print("months before the start, largest |change in the gap|:", round(float(np.abs(ES[:t0 - 1, 0]).max()), 3), "kW")

# the event-study regression: home and month dummies (from section 3) plus a TOU x month dummy for every month but 12
ks = [t for t in range(m) if t != t0 - 1]
EV = np.column_stack([(on[home] & (month == t)).astype(float) for t in ks])
be, *_ = np.linalg.lstsq(np.column_stack([EV, X[:, 1:]]), yv, rcond=None)
beta = np.insert(be[:len(ks)], t0 - 1, 0.0)
print("event-study regression against the month-by-month calculation, largest difference:", float(np.abs(beta - ES[:, 0]).max()))
assert np.allclose(beta, ES[:, 0], rtol=0, atol=1e-8)
if plt:
    k = np.arange(m) - t0
    plt.figure(figsize=(7, 3.2))
    plt.errorbar(k, ES[:, 0], yerr=1.96 * ES[:, 1], fmt="o-", ms=3, capsize=2, label="change in the gap against month 12 (simulated)")
    plt.axhline(0, color="k", lw=0.8); plt.axhline(ATT, color="g", ls="--", label=f"truth {ATT:.2f} kW")
    plt.axvspan(-0.5, k[-1] + 0.5, alpha=0.08); plt.xlabel("months from the start"); plt.ylabel("kW"); plt.legend(); plt.tight_layout(); plt.show()

for th in ["trend", "anticipate", "spill", "bundle"]:
    Q = tou_panel(T, th)
    yq, oq = Q["y"], Q["on"]
    assert (yq > 0).all(), th
    est, se = diff_means(yq[oq, t0:].mean(1) - yq[oq, :t0].mean(1), yq[~oq, t0:].mean(1) - yq[~oq, :t0].mean(1))
    pre_max = np.abs(event_study(yq, oq, t0 - 1)[:t0 - 1, 0]).max()
    assert close(est, REF["tou"]["threats"][th][0])
    print(f"{th:10s} DiD {est:+.3f} ± {1.96 * se:.3f} kW; holds the truth: {abs(est - ATT) <= 1.96 * se}; largest pre-start gap {pre_max:.3f} kW")

Only the threats that start **before** month 13 (a trend, anticipation) show in the event study. A spillover onto the comparison homes and a second change at the same time leave the months before flat and still move the estimate.

## 5. 200 redrawn panels

One seed is one draw. The page's seed (78) was picked for a calm pre-period, a teaching choice. Here, 200 new panels are drawn with NumPy's own generator (`default_rng`, vectorised, so it is quick; its numbers differ from the page's generator, the design is the same). How often does each estimator's about-95% interval hold the truth?

In [ ]:
def tou_panel_np(T, rng, noise=None, p_default=None):
    # noise: an optional (homes, months) matrix of standard-normal shocks (practice 3); p_default: put each home on the
    # tariff with this chance, whatever its engagement (an opt-out default, practice 4). Without them, the page's design.
    n, mm, t0 = T["n"], T["months"], T["start"]
    ac = (rng.random(n) < T["acShare"]).astype(float)
    s = T["size"][0] + (T["size"][1] - T["size"][0]) * rng.random(n)
    e, z = rng.standard_normal(n), rng.standard_normal(n)
    on = e + T["select"]["size"] * (s - 2) + T["select"]["noise"] * z > T["select"]["cut"]
    if p_default is not None:
        on = rng.random(n) < p_default
    tau = np.minimum(0, -(T["effect"]["base"] + T["effect"]["engaged"] * e))
    lev = T["level"]["base"] + T["level"]["ac"] * ac + T["level"]["size"] * (s - 2) + T["level"]["engaged"] * e
    t = np.arange(mm)
    season = np.array(T["season"])[t % 12]
    acs = np.array(T["acSeason"])[t % 12] * np.where(t < 12, T["cooling"][0], T["cooling"][1])
    eps = rng.standard_normal((n, mm)) if noise is None else noise
    y = lev[:, None] + season[None, :] + ac[:, None] * acs[None, :] + T["noise"] * eps
    y += (on[:, None] & (t[None, :] >= t0)) * tau[:, None]
    return y, on, tau

rng = np.random.default_rng(2026)
hold = {"naive": 0, "DiD": 0}
bias = {"naive": [], "DiD": []}
low = np.inf
for draw in range(200):
    yy, oo, tt = tou_panel_np(T, rng)
    low = min(low, yy.min())
    att = tt[oo].mean()
    pre_, post_ = yy[:, :t0].mean(1), yy[:, t0:].mean(1)
    for name, (est, se) in [("naive", diff_means(post_[oo], post_[~oo])), ("DiD", diff_means((post_ - pre_)[oo], (post_ - pre_)[~oo]))]:
        hold[name] += abs(est - att) <= 1.96 * se
        bias[name].append(est - att)
for name in hold:
    print(f"{name:6s} mean error {np.mean(bias[name]):+.3f} kW; interval held the truth in {hold[name]} of 200 draws")
print(f"lowest evening peak in the 200 panels: {low:.2f} kW")

DiD's interval holds the truth in about 95% of the draws, as it should when its assumptions hold; the naive gap's almost never does, however many homes you add.

## 6. The regions panel: a policy and solar investment

Thirty simulated regions, 2016–2025; regions that adopt a policy in 2021 get +10 MW a year of new solar. Sunnier regions adopt more, so their levels differ: DiD removes that. Then two threats: **the policy followed a boom** (adopters were already growing faster) and **projects moved across the border**: each comparison region loses 4 MW a year of projects to the adopters. Those projects are part of the adopters' +10 MW, so only the comparison regions' numbers change: the adopters' gross gain is part moved, part new.

In [ ]:
def region_panel(R, threat=None):
    r, on, Y = mulberry32(R["seed"]), [], []
    for k in range(R["n"]):
        sun = gauss(r)
        al = R["level"] + R["levelSun"] * sun
        o = 1 if sun + R["select"]["noise"] * gauss(r) > R["select"]["cut"] else 0
        y = []
        for t in range(R["years"]):
            v = al + R["growth"] * t + R["noise"] * gauss(r)
            if o and t >= R["start"]: v += R["effect"]
            if threat == "boom" and o: v += R["boom"] * (t - (R["start"] - 1))
            if threat == "border" and not o and t >= R["start"]: v += R["border"]
            y.append(v)
        on.append(o); Y.append(y)
    return np.array(Y), np.array(on, dtype=bool)

s0 = R["start"]
for th in [None, "boom", "border"]:
    Yr, onr = region_panel(R, th)
    est, se = diff_means(Yr[onr, s0:].mean(1) - Yr[onr, :s0].mean(1), Yr[~onr, s0:].mean(1) - Yr[~onr, :s0].mean(1))
    ref = REF["regions"]["did"] if th is None else REF["regions"]["threats"][th]
    assert close(est, ref[0]) and close(se, ref[1])
    pre_gap = np.abs(event_study(Yr, onr, s0 - 1)[:s0 - 1, 0]).max()
    print(f"{str(th):6s} {onr.sum()} adopters; DiD {est:+.1f} ± {1.96 * se:.1f} MW a year (truth +10); largest pre-2021 gap {pre_gap:.1f} MW")
Yb, onb = region_panel(R, "border")
gross, moved = R["effect"] * onb.sum(), -R["border"] * (~onb).sum()
print(f"border case, MW a year: gross gain in the {onb.sum()} adopters {gross}, of which moved from the {(~onb).sum()} others {moved}; new across all 30 regions {gross - moved}")
assert (Yb > 0).all()

Moved projects are a **spillover**: the comparison group is no longer the counterfactual, so DiD overstates the gain (14.0 MW a year a region, 196 in all), and the adopters' gross gain (140) is larger than what is new across all 30 regions (76; lesson A13's gross against net).

## Practice (about 90 minutes)

1. **Controls that do work.** Pretend engagement were measured: add `P["e"]` as a control in the regression of section 2. How close does it get to the truth, and why can a real analyst rarely do this?
2. **Placebo test.** Pretend the tariff started in month 7, use only months 1–12, and compute DiD. What should it be, and what is it? Repeat with the `"trend"` threat.
3. **Clusters.** Give each home a persistent shock (each month's noise carries 0.8 of the month before's) and compare the three standard errors of section 3 again. Then cluster by **month** instead of by home. Which is right, and why? *Starter code below.*
4. **Opt-out.** Change the selection rule so that every home is on the tariff with probability 0.9, whatever its engagement (a default), and redraw with `tou_panel_np(T, rng, p_default=0.9)`. What does DiD estimate now, and how does it compare with the ATE of section 1?
5. **Staggered start.** In the regions panel, let half the adopters start in 2019 and half in 2022, with an effect that grows by 2 MW a year after the start. Fit the two-way fixed-effects regression with region and year dummies. Is it +10, the average effect, or neither? (Goodman-Bacon 2021; Callaway and Sant'Anna 2021.) *Starter code below.*

In [ ]:
# Practice 3 starter: persistent shocks, then the three standard errors of section 3
def ar1_noise(rng, n, m, rho):
    # standard-normal shocks where each month carries rho of the month before (variance 1 in every month)
    e = rng.standard_normal((n, m))
    u = np.empty_like(e)
    u[:, 0] = e[:, 0]
    for t in range(1, m):
        u[:, t] = rho * u[:, t - 1] + np.sqrt(1 - rho ** 2) * e[:, t]
    return u

rng3 = np.random.default_rng(3)
y3, on3, tau3 = tou_panel_np(T, rng3, noise=ar1_noise(rng3, T["n"], T["months"], 0.8))
print({k: round(float(v), 4) for k, v in three_se(y3, on3, T["start"]).items()})
# your work: repeat over many draws and count how often each interval holds the truth (tau3[on3].mean());
# then write a standard error clustered by month (sum dd * u within each month instead of each home) and compare

In [ ]:
# Practice 5 starter: staggered starts in the regions panel
def region_panel_staggered(R, rng, starts=(3, 6), grow=2.0):
    # adopters (sunnier regions, as before) split in two: half start in 2019 (t = 3), half in 2022 (t = 6); the effect is
    # +10 MW in the first year and grows by `grow` MW a year after that. Returns Y[region, year], each region's start
    # index (-1: never adopts) and the true effect in each region-year
    n, yrs = R["n"], R["years"]
    sun = rng.standard_normal(n)
    adopt = sun + R["select"]["noise"] * rng.standard_normal(n) > R["select"]["cut"]
    start = np.full(n, -1)
    idx = np.flatnonzero(adopt)
    start[idx[: len(idx) // 2]], start[idx[len(idx) // 2:]] = starts
    t = np.arange(yrs)
    since = np.where(start[:, None] >= 0, t[None, :] - start[:, None], -1)
    effect = np.where(since >= 0, R["effect"] + grow * since, 0.0)
    Y = R["level"] + R["levelSun"] * sun[:, None] + R["growth"] * t[None, :] + R["noise"] * rng.standard_normal((n, yrs)) + effect
    return Y, start, effect

def twfe(Y, Dm):
    # y_rt = alpha_r + lambda_t + tau D_rt by least squares with region and year dummies (year 0 dropped); returns tau
    n, yrs = Y.shape
    r_id, t_id = np.repeat(np.arange(n), yrs), np.tile(np.arange(yrs), n)
    Xr = np.column_stack([Dm.ravel(), np.eye(n)[r_id], np.eye(yrs)[t_id][:, 1:]])
    return np.linalg.lstsq(Xr, Y.ravel(), rcond=None)[0][0]

Y5, start5, eff5 = region_panel_staggered(R, np.random.default_rng(5))
D5 = (start5[:, None] >= 0) & (np.arange(R["years"])[None, :] >= start5[:, None])
print("two-way fixed effects:", round(float(twfe(Y5, D5.astype(float))), 2), "MW a year")
# your work: compare with the true average effect over treated region-years (eff5[D5].mean()), and explain the gap

In [ ]:
# your work here